# Constructing an ensemble of GEMs using medusa

In [1]:
import medusa
import cobra
import numpy
from pathlib import Path
from cobra.io import read_sbml_model

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [2]:
# Load pan-Oryzae GEM
data_dir = Path(".")
data_dir = data_dir.resolve()
model_path = data_dir / "panAsp_v2.xml"
panOryzae = read_sbml_model(str(model_path.resolve()))
type(panOryzae)

Set parameter Username
Academic license - for non-commercial use only - expires 2025-09-26


cobra.core.model.Model

In [3]:
panOryzae

Name,Aspergillus_oryzae
Memory address,240cc737800
Number of metabolites,2152
Number of reactions,2250
Number of genes,1375
Number of groups,187
Objective expression,1.0*r2359 - 1.0*r2359_reverse_bde81
Compartments,"Peroxisome, Mitochondrion, Cytoplasm, Extracellular"


In [4]:
# Import BPGA table 
import pandas as pd
BPGA = pd.read_csv('BPGA2ortho_GEM_custom.csv', delimiter=";", dtype=str)

# Retain columns only for oryzae isolates and the oryzae tamplate model (as a reference)
BPGA.drop(['fumigatus','niger'], axis=1, inplace=True)
BPGA.rename(columns={'oryzae':'template'}, inplace=True)


# Many genes in the BPGA table are not used by any of the oryzae isolates
# -> remove these
model_sub = panOryzae.copy()
all_genes = [gene.id for gene in model_sub.genes]
BPGA = BPGA[BPGA['cluster'].isin(all_genes)]

# Make separate df for the isolate info and the gene info
BPGA_gene = BPGA[['cluster','present_in_n_genomes','cluster_type_manual']]
BPGA_isolates = BPGA.drop(['cluster','present_in_n_genomes','cluster_type_manual'], axis=1, inplace=False)

Read LP format model from file C:\Users\gilis\AppData\Local\Temp\tmpl4i68sd2.lp
Reading time = 0.01 seconds
: 2152 rows, 4500 columns, 17072 nonzeros


`remove_genes`: This function seems to remove genes as well as reactions catalyzed by these genes, but not metabolites. Indeed, subsequently running `prune_unused_reactions` does not alter the model. Removing unused metabolites can be achieved by running `prune_unused_metabolites`. In addition, it does not touch reactions without gene association. This is desired, however, it would also be useful to be able to remove such reactions if desired -> check which function does that 

In [5]:
from cobra import manipulation

# Takes 2min
gemList = []
for i in range(len(BPGA_isolates.columns)):
    gem_i = panOryzae.copy()
    isolate_name = BPGA_isolates.columns[i]
    genesToBeRemoved = BPGA_gene.cluster.values[BPGA_isolates[isolate_name].isna().values]
    manipulation.remove_genes(gem_i,
                              genesToBeRemoved,
                              True)
    _, _, = manipulation.prune_unused_metabolites(gem_i)
    gem_i.id = isolate_name
    gemList.append(gem_i)

# Also include the pan-model for downstream use (e.g. to gapfil from)
panOryzae.id = 'Pan_oryzae'
gemList.append(panOryzae)

Read LP format model from file C:\Users\gilis\AppData\Local\Temp\tmpbylj7xy7.lp
Reading time = 0.02 seconds
: 2152 rows, 4500 columns, 17072 nonzeros


c:\Users\gilis\OneDrive - Chalmers\Desktop\postdoc\medusa\.venv\Lib\site-packages\cobra\core\group.py:147: UserWarning: need to pass in a list
  warn("need to pass in a list")


Read LP format model from file C:\Users\gilis\AppData\Local\Temp\tmpwu3dqhsu.lp
Reading time = 0.02 seconds
: 2152 rows, 3108 columns, 10892 nonzeros
Read LP format model from file C:\Users\gilis\AppData\Local\Temp\tmpccehk12z.lp
Reading time = 0.01 seconds
: 2152 rows, 4500 columns, 17072 nonzeros
Read LP format model from file C:\Users\gilis\AppData\Local\Temp\tmpvhbpvsc4.lp
Reading time = 0.02 seconds
: 2152 rows, 3912 columns, 14436 nonzeros
Read LP format model from file C:\Users\gilis\AppData\Local\Temp\tmp0ng0jc2b.lp
Reading time = 0.01 seconds
: 2152 rows, 4500 columns, 17072 nonzeros
Read LP format model from file C:\Users\gilis\AppData\Local\Temp\tmp6hhmrxdu.lp
Reading time = 0.01 seconds
: 2152 rows, 3916 columns, 14518 nonzeros
Read LP format model from file C:\Users\gilis\AppData\Local\Temp\tmppugr43dt.lp
Reading time = 0.02 seconds
: 2152 rows, 4500 columns, 17072 nonzeros
Read LP format model from file C:\Users\gilis\AppData\Local\Temp\tmpcbq_n97d.lp
Reading time = 0.01 

In [6]:
from medusa.core import Ensemble

# takes 1min30s
testEnsemble = Ensemble(list_of_models = gemList,
                        identifier = "testID",
                        name = "testName")

Read LP format model from file C:\Users\gilis\AppData\Local\Temp\tmppf1j0bc5.lp
Reading time = 0.01 seconds
: 2152 rows, 3108 columns, 10892 nonzeros


In [7]:
# Pickle the ensemble and extracted base model
import pickle

path = "./scrap/"
pickle.dump(testEnsemble, open(path+"ensemble.pickle","wb"))
pickle.dump(gemList[0], open(path+"gem0.pickle","wb"))
pickle.dump(gemList[1], open(path+"gem1.pickle","wb"))
pickle.dump(gemList, open(path+"gemList.pickle","wb"))

In [8]:
len(testEnsemble.members)
len(testEnsemble.features)

158

1219

In [9]:
testEnsemble.members

[<Member template at 0x19d46259550>,
 <Member Aspergillus_oryzae_NBRC_4079_GCA_030270485.1 at 0x19d4ee52d20>,
 <Member Aspergillus_oryzae_3.042_GCA_000269785.2 at 0x19d4ee51820>,
 <Member Aspergillus_oryzae_TK_10GCA_009684975.1 at 0x19d4ee519a0>,
 <Member Aspergillus_oryzae_TK_27GCA_009686645.1 at 0x19d5031ec60>,
 <Member Aspergillus_oryzae_TK_41GCA_009686925.1 at 0x19d5031e5d0>,
 <Member Aspergillus_oryzae_TK_31GCA_009686725.1 at 0x19d5031f860>,
 <Member Aspergillus_oryzae_NRRL_2218 at 0x19d5031c560>,
 <Member Aspergillus_oryzae_TK_71GCA_009687525.1 at 0x19d5031e900>,
 <Member Aspergillus_oryzae_TK_42GCA_009686945.1 at 0x19d5031f770>,
 <Member Aspergillus_oryzae_TK_20GCA_009686505.1 at 0x19d5031f6e0>,
 <Member Aspergillus_oryzae_NRRL_459 at 0x19d5031db20>,
 <Member Aspergillus_oryzae_TK_14GCA_009685055.1 at 0x19d5031fe00>,
 <Member Aspergillus_oryzae_NRRL_466 at 0x19d5031c050>,
 <Member Aspergillus_oryzae_NRRL_5589 at 0x19d5031dbb0>,
 <Member Aspergillus_oryzae_TK_39GCA_009686885.1 at

Compare object size of ensemble with list of individual models, both in terms of RAM usage to load the data object and file size upon exporting the data object.

In [ ]:
import os
import psutil
from copy import deepcopy

# RAM required to load the ensemble
RAM_before = psutil.Process(os.getpid()).memory_info()[0]/1024**2 # Units = MB
testEnsemble_copy = deepcopy(testEnsemble)
RAM_after = psutil.Process(os.getpid()).memory_info()[0]/1024**2 # Units = MB
RAM_used = RAM_after - RAM_before
print("%.2f" % (RAM_used), "MB")

# RAM required to load a single GEM
RAM_before = psutil.Process(os.getpid()).memory_info()[0]/1024**2 # Units = MB
gemList_copy = deepcopy(gemList[0])
RAM_after = psutil.Process(os.getpid()).memory_info()[0]/1024**2 # Units = MB
RAM_used = RAM_after - RAM_before
print("%.2f" % (RAM_used), "MB")

-127.41 MB
-27.66 MB


Running the code multiple times shows that the reported memory usage "randomly" varies across repeated runs. However, most of the runs indicate a memory usage for the ensemble of approximately 29MB, while loading a single GEM takes approximately 17.5MB. As such, sequentially loading all individual models is expected to require a total of 17.5*156=2730MB, or nearly 3GB. In other words, we working with medusa leads to a 100-fold reduction in RAM usage.

EDIT: since I've update my code to include the oryzae template as an ensemble member, these numbers changed drastically. With additionally changing values per run, it doesn't look a particularly reliable benchmark to me...

In [ ]:
# Pickle the ensemble and extracted base model
import pickle

path = "./scrap/"
pickle.dump(testEnsemble, open(path+"ensemble.pickle","wb"))
pickle.dump(gemList[1], open(path+"gem1.pickle","wb"))
pickle.dump(gemList, open(path+"gemList.pickle","wb"))

# Check for file size of ensemble
file_path = "./scrap/ensemble.pickle"
file_info = os.stat(file_path)
mb = file_info.st_size/(1024.0**2) # Convert from bytes to MB
print("%.2f %s" % (mb, 'MB for a 157 member ensemble'))

# Check for file size of a single GEM
file_path = "./scrap/gem0.pickle"
file_info = os.stat(file_path)
mb = file_info.st_size/(1024.0**2) # Convert from bytes to MB
print("%.2f %s" % (mb, 'MB per model'))
print("%.2f" % (mb*157),'MB for 157 individual model files (estimated).')

# Check for file size of a list of GEMs
file_path = "./scrap/gemList.pickle"
file_info = os.stat(file_path)
mb = file_info.st_size/(1024.0**2) # Convert from bytes to MB
print("%.2f %s" % (mb, 'MB for a list with 157 members'))

6.85 MB for a 156 member ensemble
1.39 MB per model
218.93 MB for 157 individual model files (estimated).
16.74 MB for a list with 157 members


We also gain in terms of storage. The ensemble with 156 members requires 6.55MB of storage. An individual GEM requires 1.64MB, so storing each model individually is estimated to require 256MB of storage. Storing a list with all individual models requires 158MB.

EDIT: after changing to include the oryzae template model, storing a list with all individual models requires 17MB in stead of 158MB. Unclear why...